# 02 — Transcripción en GPU

**Qué hace.** Transcribe el corpus con faster-whisper en una GPU de Colab y mide lo que pide la Fase 2 de Task 1: dos tamaños de modelo, CPU contra GPU, float16 contra int8, VRAM pico y el efecto del glosario.

**Qué lee.** Los 10 audios desde una carpeta privada de Drive. No están en el repo, y se ubican por su nombre original (`config.yaml → audios`), las referencias `data/referencias/*.txt` y los parámetros de `config.yaml → asr`.

**Qué produce.** La caché de transcripciones en `rutas.transcripciones_dir`, que se commitea para que la app y CI nunca re-transcriban, y las tablas de WER, CER, segundos por minuto de audio y VRAM en `rutas.resultados_dir`.

**Dónde corre.** Google Colab con GPU (Entorno de ejecución → Cambiar tipo → T4). Llama a `src.transcripcion.transcribir` y a `src.metricas`; no reimplementa nada.

## 0. Preparar Colab

Clona el repositorio e instala las dependencias fijadas. La URL del repo es lo único escrito aquí: hace falta antes de que exista `config.yaml` en la máquina.

In [ ]:
!git clone https://github.com/mmlis02/HW_05_202602.git
%cd HW_05_202602
!pip install -q -r requirements.txt

In [ ]:
import sys
from pathlib import Path

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())
sys.path.insert(0, str(RAIZ))
from src.config import cargar_config

cfg = cargar_config()
rutas = cfg["rutas"]

## 1. Montar Drive y ubicar los audios

Los audios son privados: no están en el repositorio, y el enlace a su carpeta tampoco. Se ubican por su **nombre original** en Drive ("task 1-1" … "task 2-5"), sin renombrarlos. El mapeo nombre → clip está en `config.yaml → audios.archivos`, y desde aquí todo el notebook usa los clips `e1 … l5`.

`src.audios.buscar_audios` recorre `audios.drive_raiz` y todas sus subcarpetas. Acepta cualquier extensión de `audios.extensiones` y no distingue mayúsculas. **Se detiene** si falta alguno de los 10 o si alguno aparece más de una vez, y dice cuáles por su nombre original. En un Drive con muchos archivos, el recorrido puede tardar uno o dos minutos.

In [ ]:
from google.colab import drive

drive.mount(cfg["audios"]["drive_montaje"])

In [ ]:
from src.audios import buscar_audios

audios = buscar_audios(cfg["audios"]["drive_raiz"])
carpetas = sorted({str(ruta.parent) for ruta in audios.values()})
if len(carpetas) == 1:
    print(f"Carpeta encontrada: {carpetas[0]}")
else:
    print("Los audios están en varias carpetas:", *carpetas, sep="\n  ")
for clip, ruta in audios.items():
    print(f"  {clip}  ←  {ruta.name}")

**Confirma que la carpeta de arriba es la de tus audios antes de seguir.** Si aparecen varias carpetas, revisa que no haya una copia antigua con los mismos nombres.

*Pendiente: medir la duración real de cada audio (2 a 4 minutos) y reemplazar con ella la estimación por palabras de `01_ground_truth`.*

## 2. Prueba de que se usó la GPU

Device leído de la librería en tiempo de ejecución (no de `config.yaml`), `nvidia-smi` y VRAM medida. CPU y GPU dan el mismo texto; solo la medición los separa.

*Pendiente.*

## 3. Modelos

`small` contra `large-v3` (`asr.modelos_comparacion`) sobre los mismos clips: WER, CER, segundos por minuto de audio y VRAM pico.

*Pendiente.*

## 4. CPU contra GPU

Mismos clips y mismo modelo en los dos devices. Tiempo, y si el texto cambió.

*Pendiente.*

## 5. float16 contra int8

Precisiones de `asr.precisiones_comparacion`. Reportar si la transcripción cambió, no solo si fue más rápida: más rápido y peor no es una mejora.

*Pendiente.*

## 6. VRAM

VRAM pico por combinación de modelo y precisión. Referencia del enunciado: large a int8 ≈ 2 926 MB contra 4 525 MB a float16.

*Pendiente.*

## 7. Glosario

Transcribir sin glosario y con `asr.glosario`. Resolver en la documentación de la versión instalada qué mecanismo existe (`initial_prompt`, `hotwords`) y medir la diferencia de WER con intervalo de confianza.

*Pendiente.*

## 8. Guardar la caché

Escribir las transcripciones (texto y palabras con probabilidad) en `rutas.transcripciones_dir` y commitearlas.

*Pendiente.*